In [1]:
# import packages
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as mpl
from tqdm import tqdm
from collections import Counter

In [2]:
# the goal of this notbeook is collapse TF matches to a single call and count top motifs for each cell type
# the merged bed files loaded here were generated by 'mpac_promoter_seqlet_bedmap_merge.sh'
# the 'rules' are as follows:
# 1. single intervals are unchanged
# 2. merged intervals that have the same TF call are just called that TF
# 3. merged intervals that have multiple TF calls will be called the TF with the highest absolute attribution in the interval

In [5]:
def assign_representative_tf(bed_file):
    """
    Reads a mapped BED file, assigns a single representative TF, and
    classifies its activity based on attribution score.

    Args:
        bed_file (pd.DataFrame): DataFrame from the bedmap output.

    Returns:
        pd.DataFrame: The original DataFrame with two new columns:
                      'representative_tf' and 'activity_class'.
    """
    df = bed_file
    data_column_index = 3

    representative_tfs = []
    activity_classes = [] # New list to store the activity class
    attributions = []
    for index, row in df.iterrows():
        mapped_info_string = str(row[data_column_index])
        tf_hits = mapped_info_string.split(';')
        
        best_tf = ''
        best_score = 0.0 # Variable to hold the signed score of the winner
        
        if len(tf_hits) == 1:
            # RULE 1: Only one TF is present
            parts = tf_hits[0].split('_')
            best_tf = ('_').join(parts[:2])
            try:
                best_score = float(parts[-1])
            except (ValueError, IndexError):
                best_score = 0.0 # Default if score is missing
        else:
            # RULES 2 & 3: Multiple TFs are present
            max_abs_attribution = -1.0
            all_tf_names = set()
            
            for hit in tf_hits:
                parts = hit.split('_')
                tf_name = ('_').join(parts[:2])
                all_tf_names.add(tf_name)
                
                try:
                    attribution_score = float(parts[-1])
                    if abs(attribution_score) > max_abs_attribution:
                        max_abs_attribution = abs(attribution_score)
                        best_tf = tf_name
                        best_score = attribution_score # Store the signed score
                except (ValueError, IndexError):
                    continue
            
            if len(all_tf_names) == 1:
                # RULE 2: Motifs are the same, just use the single name
                best_tf = list(all_tf_names)[0]

        # Append the chosen TF and determine its class
        representative_tfs.append(best_tf)
        
        if best_score > 0:
            activity_classes.append('Activator')
            attributions.append(best_score)
        elif best_score < 0:
            activity_classes.append('Repressor')
            attributions.append(best_score)
        else:
            activity_classes.append('Neutral') # For scores of 0 or errors
            attributions.append(best_score)

    # Add the two new columns to the DataFrame
    df['representative_tf'] = representative_tfs
    df['activity_class'] = activity_classes
    df['attribution'] = attributions
    
    return df

In [6]:
def assign_representative_tf_parm(bed_file):
    """
    Reads a mapped BED file, assigns a single representative TF, and
    classifies its activity based on attribution score.

    Args:
        bed_file (pd.DataFrame): DataFrame from the bedmap output.

    Returns:
        pd.DataFrame: The original DataFrame with two new columns:
                      'representative_tf' and 'activity_class'.
    """
    df = bed_file
    data_column_index = 3

    representative_tfs = []
    activity_classes = [] # New list to store the activity class
    attributions = []
    for index, row in df.iterrows():
        mapped_info_string = str(row[data_column_index])
        tf_hits = mapped_info_string.split(';')
        
        best_tf = ''
        best_score = 0.0 # Variable to hold the signed score of the winner
        
        if len(tf_hits) == 1:
            # RULE 1: Only one TF is present
            parts = tf_hits[0].split(':')
            best_tf = parts[0].split('-')[0]
            try:
                best_score = float(parts[1])
            except (ValueError, IndexError):
                best_score = 0.0 # Default if score is missing
        else:
            # RULES 2 & 3: Multiple TFs are present
            max_abs_attribution = -1.0
            all_tf_names = set()
            
            for hit in tf_hits:
                parts = hit.split(':')
                tf_name = parts[0].split('-')[0]
                all_tf_names.add(tf_name)
                
                try:
                    attribution_score = float(parts[1])
                    if abs(attribution_score) > max_abs_attribution:
                        max_abs_attribution = abs(attribution_score)
                        best_tf = tf_name
                        best_score = attribution_score # Store the signed score
                except (ValueError, IndexError):
                    continue
            
            if len(all_tf_names) == 1:
                # RULE 2: Motifs are the same, just use the single name
                best_tf = list(all_tf_names)[0]

        # Append the chosen TF and determine its class
        representative_tfs.append(best_tf)
        
        if best_score > 0:
            activity_classes.append('Activator')
            attributions.append(best_score)
        elif best_score < 0:
            activity_classes.append('Repressor')
            attributions.append(best_score)
        else:
            activity_classes.append('Neutral') # For scores of 0 or errors
            attributions.append(best_score)

    # Add the two new columns to the DataFrame
    df['representative_tf'] = representative_tfs
    df['activity_class'] = activity_classes
    df['attribution'] = attributions
    
    return df

In [3]:
# 9:55PM 092125 - for now we will write all the code for the no-overlap bed file but we'll make it a function so you can plug and chug for comparisons
# MPAC #
# open K562 no min overlap bed file
k562_merged_bed_noMin = pd.read_csv('../processed_data/bed_files/K562_bedOps_merged_noMin_seqlets_01_112225.bed', sep = '\t', header = None)
# hepg2 no min overlap bed file
hepg2_merged_bed_noMin = pd.read_csv('../processed_data/bed_files/HepG2_bedOps_merged_noMin_seqlets_01_112225.bed', sep = '\t', header = None)
# sknsh no min overlap bed file
sknsh_merged_bed_noMin = pd.read_csv('../processed_data/bed_files/SKNSH_bedOps_merged_noMin_seqlets_01_112225.bed', sep = '\t', header = None)

In [4]:
# open PARM no min overlap bed files - PARM TF Calls
# k562
k562_parm_merged_noMin = pd.read_csv('../processed_data/bed_files/PARM_K562_sig_TFs_bedOps_merged_noMin_112225.bed', sep = '\t', header = None)
# hepg2
hepg2_parm_merged_noMin = pd.read_csv('../processed_data/bed_files/PARM_HepG2_sig_TFs_bedOps_merged_noMin_112225.bed', sep = '\t', header = None)

In [10]:
# open PARM tangermeme call bed files
# k562
k562_parm_tangermeme_merged = pd.read_csv('../processed_data/bed_files/parm_k562_tangermeme_bedOps_merged_noMin_seqlets_01_112225.bed', sep = '\t', header = None)
# hepg2
hepg2_parm_tangermeme_merged = pd.read_csv('../processed_data/bed_files/parm_hepg2_tangermeme_bedOps_merged_noMin_seqlets_01_112225.bed', sep = '\t', header = None)

In [11]:
# open promoterAI call bed file
pAI_tangermeme_merged = pd.read_csv('../processed_data/bed_files/promoterAI_bedOps_merged_noMin_seqlets_01.bed', sep = '\t', header = None)

In [12]:
# collapse TFs for each cell type - MPAC
# k562
k562_noMin_collapsed = assign_representative_tf(k562_merged_bed_noMin)
# hepg2
hepg2_noMin_collapsed = assign_representative_tf(hepg2_merged_bed_noMin)
# sknsh
sknsh_noMin_collapsed = assign_representative_tf(sknsh_merged_bed_noMin)

In [13]:
# save mpac BED files to disk
k562_noMin_collapsed.to_csv('../processed_data/bed_files/mpac_k562_merged_collapsed_repTFs_112225.bed', sep = '\t', index = False, header = None)
hepg2_noMin_collapsed.to_csv('../processed_data/bed_files/mpac_hepg2_merged_collapsed_repTFs_112225.bed', sep = '\t', index = False, header = None)
sknsh_noMin_collapsed.to_csv('../processed_data/bed_files/mpac_sknsh_merged_collapsed_repTFs_112225.bed', sep = '\t', index = False, header = None)

In [14]:
# collapse PARM TF match calls
# k562
k562_parmCall_merged = assign_representative_tf_parm(k562_parm_merged_noMin)
# hepg2
hepg2_parmCall_merged = assign_representative_tf_parm(hepg2_parm_merged_noMin)

In [15]:
# save collapsed PARM Call BED files to disk
k562_parmCall_merged.to_csv('../processed_data/bed_files/parm_parmCalled_k562_merged_collapsed_repTFs_112225.bed', sep = '\t', index = False, header = None)
hepg2_parmCall_merged.to_csv('../processed_data/bed_files/parm_parmCalled_hepg2_merged_collapsed_repTFs_112225.bed', sep = '\t', index = False, header = None)

In [16]:
# collapse PARM tangermeme calls
# k562
parm_k562_tmeme_collapsed = assign_representative_tf(k562_parm_tangermeme_merged)
# hepg2
parm_hepg2_tmeme_collapsed = assign_representative_tf(hepg2_parm_tangermeme_merged)

In [28]:
# save collapsed PARM Call BED files to disk
parm_k562_tmeme_collapsed.to_csv('../processed_data/bed_files/parm_tangermeme_k562_merged_collapsed_repTFs_112225.bed', sep = '\t', index = False, header = None)
parm_hepg2_tmeme_collapsed.to_csv('../processed_data/bed_files/parm_tangermeme_hepg2_merged_collapsed_repTFs_112225.bed', sep = '\t', index = False, header = None)

In [30]:
# collapse promoterAI tangermeme calls
pAI_tmeme_collapsed = assign_representative_tf(pAI_tangermeme_merged)

In [32]:
# save collapsed promoterAI BED files to disk
pAI_tmeme_collapsed.to_csv('../processed_data/bed_files/promoterAI_tangermeme_merge_collapsed_repTFs_112225.bed', sep = '\t', index = False, header = None)